# Session 04 · Part 3 — Context optimization for long tasks

Long pages and long conversations fill the context. Deep Agents compresses it automatically:

- **Offloading**: a tool result above a token limit (default 20,000) is saved to `/large_tool_results/<id>`; the context keeps
  the path and a preview.
- **Summarization**: above a trigger (default 85% of the model's input window) old messages are replaced by a summary;
  the original messages are saved to `/conversation_history/<id>.md`.

The defaults are far above what three questions produce, so this notebook **lowers both thresholds** by passing its own
`FilesystemMiddleware` and `SummarizationMiddleware` (they replace the built-in instances with the same name).
Then the agent keeps a **progress file** so that a new thread can continue the work.

<img src="images/da_offloading_results.png" alt="offloading" width="360"> <img src="images/da_summarization.png" alt="summarization" width="480">

*Figures: LangChain Deep Agents docs, Context engineering.*

Docs: [Context engineering](https://docs.langchain.com/oss/python/deepagents/context-engineering) ·
[Override a default middleware](https://docs.langchain.com/oss/python/deepagents/customization#override-a-default-middleware-instance)

## 0. Setup

In [1]:
import logging
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
logger = logging.getLogger(__name__)

# ── Keys and model (.env of this session folder) ─────────────────────────────
load_dotenv()
for name in ("OPENAI_API_KEY", "TAVILY_API_KEY"):
    if not os.getenv(name, "").strip():
        raise RuntimeError(f"Missing {name}: copy .env.example to .env in the session folder and fill it in")
model = init_chat_model(os.getenv("MODEL", "openai:gpt-6-luna"))
logger.info("Model: %s", model.model_name)

OUT = Path("output")  # everything the agent writes goes here (git-ignored)
OUT.mkdir(exist_ok=True)

INFO | Model: gpt-6-luna


In [2]:
from utils import context_window, read_page, routes, stream, system_prompt  # notebooks/utils.py

WINDOW = context_window(model)  # max input tokens, from the model's LangChain profile
logger.info("Context window: %s tokens", f"{WINDOW:,}")

INFO | Context window: 1,050,000 tokens


## 1. Two agents, same task
- **baseline**: offloading switched off (`tool_token_limit_before_evict=None`), default summarization (never reached here).
- **managed**: offload results above 1,500 tokens, summarize above 6,000 tokens and keep the last 6 messages
  (`trim_tokens_to_summarize=None` lets the summarizer read all older messages, as `create_summarization_middleware` does).

Both use the same backend layout: thread state by default, `/workspace/` on disk (`output/workspace-part3/`).

In [3]:
from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend
from deepagents.middleware import FilesystemMiddleware, SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

WORKSPACE = OUT / "workspace-part3"
WORKSPACE.mkdir(exist_ok=True)
(WORKSPACE / "progress.md").unlink(missing_ok=True)  # start clean

def layout(purpose=None) -> dict:
    """/workspace/ on disk; purpose: what the agent keeps there (None: nothing, no # Files section)."""
    return {"/workspace/": (FilesystemBackend(root_dir=WORKSPACE.resolve(), virtual_mode=True), purpose)}


def make_agent(offload_limit, summarize_at=None, workspace_purpose=None):
    """Same agent, different context settings. Passing our own FilesystemMiddleware / SummarizationMiddleware
    replaces the built-in instance with the same name (order in the list does not matter)."""
    files = layout(workspace_purpose)
    backend = CompositeBackend(default=StateBackend(), routes=routes(files))
    middleware = [FilesystemMiddleware(backend=backend, tool_token_limit_before_evict=offload_limit)]
    if summarize_at:
        middleware.append(SummarizationMiddleware(model, backend=backend, trigger=("tokens", summarize_at),
                                                  keep=("messages", 6), trim_tokens_to_summarize=None))
    return create_deep_agent(model=model, tools=[read_page], system_prompt=system_prompt(files), backend=backend,
                             middleware=middleware, checkpointer=InMemorySaver())


baseline = make_agent(offload_limit=None)
managed = make_agent(offload_limit=1500, summarize_at=6000)

QUESTIONS = [
    "Read https://en.wikipedia.org/wiki/Heat_pump and list 3 facts about efficiency.",
    "Now read only https://en.wikipedia.org/wiki/Air_source_heat_pump and compare it with ground-source heat pumps in 3 bullets.",
]


def run(agent, thread_id: str):
    """Ask the questions in one thread; print one context row per model call."""
    totals = []
    for q in QUESTIONS:
        print("Q:", q)
        _, log = stream(agent, q, thread_id, window=WINDOW)
        log.table()
        totals += [log.sizes(i)["total"] for i in range(len(log.calls))]
    return totals, log

## 2. Baseline: every page stays in the context

In [4]:
base_rows, base_log = run(baseline, "base-1")

Q: Read https://en.wikipedia.org/wiki/Heat_pump and list 3 facts about efficiency.
→ read_page({"url": "https://en.wikipedia.org/wiki/Heat_pump"})


INFO | read_page('https://en.wikipedia.org/wiki/Heat_pump') -> ok 1563 ms


← read_page success: [Jump to content](#bodyContent)

[![](/static/images/icons/enwiki-25.svg)  ![Wikipedia](/static/images/mobile/…
1. Heat pumps can deliver about 1–4.5 kWh of heat per 1 kWh of electricity. [https://en.wikipedia.org/wiki/Heat_pump](https://en.wikipedia.org/wiki/Heat_pump)
2. A well-designed heat pump typically has a coefficient of performance (COP) of 3–5 under the page’s stated example conditions (10°C outdoors and 20°C indoors). [https://en.wikipedia.org/wiki/Heat_pump](https://en.wikipedia.org/wiki/Heat_pump)
3. Heat pumps perform most efficiently when the temperature difference between the heat source and destination is small; for heating, efficiency generally declines as outdoor temperatures fall. [https://en.wikipedia.org/wiki/Heat_pump](https://en.wikipedia.org/wiki/Heat_pump)
call messages   tokens  in this call
   1        1    2,156  -
   2        3   48,754  -
Q: Now read only https://en.wikipedia.org/wiki/Air_source_heat_pump and compare it with ground-so

INFO | read_page('https://en.wikipedia.org/wiki/Air_source_heat_pump') -> ok 317 ms


← read_page success: ![](/static/images/icons/enwiki-25.svg)
![Wikipedia](/static/images/mobile/copyright/wikipedia-wordmark-en-25.…
- **Heat source:** Air-source heat pumps draw heat from outdoor air; ground-source heat pumps use heat from the ground via underground pipes. [https://en.wikipedia.org/wiki/Air_source_heat_pump](https://en.wikipedia.org/wiki/Air_source_heat_pump)
- **Cold-weather performance:** Ground-source systems are more efficient in cold conditions because ground temperatures are more stable, while air-source heat pumps become less efficient as outdoor temperatures drop. [https://en.wikipedia.org/wiki/Air_source_heat_pump](https://en.wikipedia.org/wiki/Air_source_heat_pump)
- **Installation cost:** Air-source heat pumps generally cost less to install; ground-source systems cost more because installing their ground loops requires excavation. [https://en.wikipedia.org/wiki/Air_source_heat_pump](https://en.wikipedia.org/wiki/Air_source_heat_pump)
call messages   tokens 

## 3. Managed: offloading and summarization

In [5]:
man_rows, man_log = run(managed, "managed-1")
print(f"\npeak context: baseline {max(base_rows):,} tokens, managed {max(man_rows):,} tokens")

Q: Read https://en.wikipedia.org/wiki/Heat_pump and list 3 facts about efficiency.
→ read_page({"url": "https://en.wikipedia.org/wiki/Heat_pump"})


INFO | read_page('https://en.wikipedia.org/wiki/Heat_pump') -> ok 494 ms


← read_page success: Tool result too large, the result of this tool call call_hs2aEpJfDTbCX1wkWGT8Vaey was saved in the filesystem …
→ grep({"pattern": "efficien", "path": "/large_tool_results/call_hs2aEpJfDTbCX1wkWGT8Vaey", "glob": null, "output_mod…)
← grep success: /large_tool_results/call_hs2aEpJfDTbCX1wkWGT8Vaey:
  27:   + [6.1 Improvement of coefficient of performance by…
- For 1 kWh of electricity, heat pumps can transfer about 1–4.5 kWh of thermal energy into a building. [https://en.wikipedia.org/wiki/Heat_pump](https://en.wikipedia.org/wiki/Heat_pump)
- A heat pump’s coefficient of performance (COP) is greater than 1, making it more efficient for heating than electric resistance heating. [https://en.wikipedia.org/wiki/Heat_pump](https://en.wikipedia.org/wiki/Heat_pump)
- COP decreases as the temperature of the heated reservoir rises, so more work is needed to transfer each unit of heat. [https://en.wikipedia.org/wiki/Heat_pump](https://en.wikipedia.org/wiki/Heat_pump)
call mes

INFO | read_page('https://en.wikipedia.org/wiki/Air_source_heat_pump') -> ok 318 ms


← read_page success: Tool result too large, the result of this tool call call_QkP3OpaMQKlLkKDAzqx8Pv6D was saved in the filesystem …
→ grep({"pattern": "ground", "path": "/large_tool_results/call_QkP3OpaMQKlLkKDAzqx8Pv6D", "glob": null, "output_mode"…)
← grep success: /large_tool_results/call_QkP3OpaMQKlLkKDAzqx8Pv6D:
  64: ASHPs are the most common type of heat pump and, usua…
- **Installation cost:** Air-source heat pumps generally cost less to install because ground-source systems require excavation for underground loops. [https://en.wikipedia.org/wiki/Air_source_heat_pump](https://en.wikipedia.org/wiki/Air_source_heat_pump)
- **Cold-weather efficiency:** Ground-source heat pumps are generally more efficient in cold conditions; their ground heat source stays more stable than outdoor air, so their COP varies less as air temperatures change. [https://en.wikipedia.org/wiki/Air_source_heat_pump](https://en.wikipedia.org/wiki/Air_source_heat_pump)
- **Air-source performance:** In mild we

### What the model saw instead of the page
The offloaded result is a short message: where the file is, how to read it in parts, and a preview.

In [6]:
state = managed.get_state({"configurable": {"thread_id": "managed-1"}}).values
offload = next(m.text for m in state["messages"] if m.type == "tool" and "Tool result too large" in m.text)
print(offload[:700])

Tool result too large, the result of this tool call call_hs2aEpJfDTbCX1wkWGT8Vaey was saved in the filesystem at this path: /large_tool_results/call_hs2aEpJfDTbCX1wkWGT8Vaey

You can read the result from the filesystem by using the read_file tool, but make sure to only read part of the result at a time.

You can do this by specifying an offset and limit in the read_file tool call. For example, to read the first 100 lines, you can use the read_file tool with offset=0 and limit=100.

Here is a preview showing the head and tail of the result (lines of the form `... [N lines truncated] ...` indicate omitted lines in the middle of the content):

1  [Jump to content](#bodyContent)
2  
3  [![](/sta


### Files written by the harness
`/large_tool_results/…` holds the full pages; `/conversation_history/…` holds the messages replaced by the summary.
Both are in the thread state (the default backend), so they belong to this thread only.

In [7]:
for path in sorted(state["files"]):
    print(path)
summary = next((m.text for m in man_log.messages if m.additional_kwargs.get("lc_source") == "summarization"), None)
print("\nfirst message of the last call:\n", (summary or "no summary yet")[:600])

/conversation_history/session_4a6f87769ca34b9ab286fe8a579bb5f4.md
/large_tool_results/call_QkP3OpaMQKlLkKDAzqx8Pv6D
/large_tool_results/call_hs2aEpJfDTbCX1wkWGT8Vaey

first message of the last call:
 You are in the middle of a conversation that has been summarized.

The full conversation history has been saved to /conversation_history/session_4a6f87769ca34b9ab286fe8a579bb5f4.md should you need to refer back to it for details.

A condensed summary follows:

<summary>
## SESSION INTENT
Read the Wikipedia article on heat pumps and give the user three accurate, relevant facts about their efficiency.

## SUMMARY
- Source: https://en.wikipedia.org/wiki/Heat_pump. Its fetched text is stored in the artifact below.
- A search for “efficien” found these relevant article facts:
  1. Heat pumps can tr


## 4. A progress file for the next thread
Offloading and summarization stay on (higher limits); the `<files>` section of the prompt now says what `/workspace/progress.md` is for (on disk).
Step 1 runs in thread `task-1`. Thread `task-2` starts with no messages and is told only to continue.

In [8]:
PROGRESS = ("progress.md tracks tasks with several steps: a Done list and a Next list with every open step and its urls, "
            "including steps planned for a later session. Update it before you answer; when asked to continue a task, "
            "read it first and do the first Next item")
worker = make_agent(offload_limit=5000, summarize_at=12000, workspace_purpose=PROGRESS)  # higher limits: reads of the
# offloaded page stay readable, so the demo shows the progress file, not repeated reads
print(system_prompt(layout(PROGRESS)).split("<files>")[1])

TASK = ("Task in two steps. Step 1: read https://en.wikipedia.org/wiki/Electric_vehicle_battery and note 3 facts about "
        "battery lifetime. Step 2: read https://en.wikipedia.org/wiki/Battery_recycling and add 3 facts about recycling. "
        "Do step 1 now and record step 2 under Next; step 2 will be done in the next session.")
r1, _ = stream(worker, TASK, "task-1", window=WINDOW)
print("\n--- progress.md ---\n" + (WORKSPACE / "progress.md").read_text())

.
- If the request is unclear, ask one short question.
</instructions>

<output_description>
Answer quick questions in 1 to 3 sentences. Put each source url in brackets after the sentence it supports.
</output_description>


→ read_page({"url": "https://en.wikipedia.org/wiki/Electric_vehicle_battery"})
→ ls({"path": "/workspace"})
← ls success: No files found


INFO | read_page('https://en.wikipedia.org/wiki/Electric_vehicle_battery') -> ok 388 ms


← read_page success: Tool result too large, the result of this tool call call_bvCe36IBBaEXBIfGP0r05pip was saved in the filesystem …
→ grep({"pattern": "Lifetime", "path": "/large_tool_results", "glob": "call_bvCe36IBBaEXBIfGP0r05pip", "output_mode":…)
← grep success: No matches found
→ read_file({"file_path": "/large_tool_results/call_bvCe36IBBaEXBIfGP0r05pip", "offset": 0, "limit": 240})
← read_file success: [Output was truncated due to size limits. The file content is very large. Consider reformatting the file to ma…
→ read_file({"file_path": "/large_tool_results/call_bvCe36IBBaEXBIfGP0r05pip", "offset": 540, "limit": 100})
→ read_file({"file_path": "/workspace/progress.md", "offset": 0, "limit": 100})
← read_file success: @@ lines 541-640 of 805 | next offset 640 @@
In addition, climatic chambers control environmental conditions d…
← read_file error: Error: File '/progress.md' not found
→ grep({"pattern": "## Battery life", "path": "/large_tool_results", "glob": "call_bvCe36IBBaEXB

INFO | read_page('https://en.wikipedia.org/wiki/Electric_vehicle_battery') -> ok 388 ms


← read_page success: Tool result too large, the result of this tool call call_0Y5hJkzUOznvviuTwQ9fEmEG was saved in the filesystem …
→ write_file({"file_path": "/workspace/progress.md", "content": "# Progress\n\n## Done\n- Step 1: Read the “Battery life” s…)
← write_file success: Updated file /workspace/progress.md
- With suitable power supplies, good battery lifespan is usually achieved at charging rates no higher than 0.5C, meaning a full charge takes at least two hours. [https://en.wikipedia.org/wiki/Electric_vehicle_battery]
- Geotab data cited in the article found that the average EV battery retained 90% of its initial capacity after six years and six months in service. [https://en.wikipedia.org/wiki/Electric_vehicle_battery]
- The article reports a December 2024 study suggesting EV batteries in real-world conditions may last up to one-third longer than laboratory estimates. [https://en.wikipedia.org/wiki/Electric_vehicle_battery]

Step 2, on battery recycling, is recorded under *

In [9]:
r2, _ = stream(worker, "Continue the battery task.", "task-2", window=WINDOW)
calls = [(tc["name"], tc["args"].get("file_path") or tc["args"].get("url")) for m in r2["messages"] if m.type == "ai" for tc in m.tool_calls]
print("tool calls in task-2:", calls)
print("\n--- progress.md ---\n" + (WORKSPACE / "progress.md").read_text())

→ read_file({"file_path": "/workspace/progress.md", "offset": 0, "limit": 200})
← read_file success: @@ lines 1-12 of 12 @@
# Progress

## Done
- Step 1: Read the “Battery life” section of Wikipedia’s “Electric …
→ read_page({"url": "https://en.wikipedia.org/wiki/Battery_recycling"})


INFO | read_page('https://en.wikipedia.org/wiki/Battery_recycling') -> ok 364 ms


← read_page success: Tool result too large, the result of this tool call call_VmzaxfOXUA8p6eHZJizXOub8 was saved in the filesystem …
→ read_file({"file_path": "/large_tool_results/call_VmzaxfOXUA8p6eHZJizXOub8", "offset": 0, "limit": 180})
← read_file success: [Output was truncated due to size limits. The file content is very large. Consider reformatting the file to ma…
→ read_file({"file_path": "/large_tool_results/call_VmzaxfOXUA8p6eHZJizXOub8", "offset": 47, "limit": 100})
← read_file success: [Output was truncated due to size limits. The file content is very large. Consider reformatting the file to ma…
→ ls({"path": "/workspace"})
← ls success: ['/workspace/progress.md']
→ read_file({"file_path": "/workspace/progress.md", "offset": 0, "limit": 100})
← read_file success: @@ lines 1-12 of 12 @@
# Progress

## Done
- Step 1: Read the “Battery life” section of Wikipedia’s “Electric …
→ read_file({"file_path": "/conversation_history/session_9b32f2833cb54234b2751c9e896cdd20.md", "offset"

### Try it

In [10]:
question = "Read https://en.wikipedia.org/wiki/Solar_water_heating and list 3 facts."  # your own request
# result, log = stream(managed, question, "try-1", window=WINDOW); log.table()

### Optional
- Let the agent compact on demand: add `create_summarization_tool_middleware(model, backend)` and it gets a
  `compact_conversation` tool (docs: Context engineering → On-demand compaction tool).
- Change `offload_limit` and `summarize_at` and watch the table.

### Done when
- The managed run has an offload message, files under `/large_tool_results/` and `/conversation_history/`, and a lower peak.
- `task-2` read `/workspace/progress.md` before reading any page and finished step 2.